# Scenario 1 — Puzzletron: Heterogeneous Pruning (Comparison) (~6h on 2x H200)

This notebook compresses Qwen3-8B to ~7B parameters using **Puzzletron** (heterogeneous NAS-based pruning), then distills and evaluates the result.

This serves as the **comparison run** for Scenario 1. The recommended approach for moderate compression is Minitron (see [`scenario1_minitron.ipynb`](scenario1_minitron.ipynb)). We run Puzzletron here to demonstrate how it compares at this compression level.

**Pipeline:** Install → Prepare calibration data → Configure → NAS search → Evaluate → Patch → Distill → Evaluate

**Prerequisites:**
- Run [`00_prerequisites.ipynb`](00_prerequisites.ipynb) first to prepare the distillation dataset.
- Base model downloaded at `/workspace/models/Qwen3-8B`.

## 1. Install Puzzletron dependencies

In [ ]:
!cd /workspace/Model-Optimizer && \
pip install -e .[hf,puzzletron] && \
pip install -r examples/puzzletron/requirements.txt

## 2. Prepare calibration dataset

Puzzletron requires explicit dataset preparation. We download and format the [Nemotron-Post-Training-Dataset-v2](https://huggingface.co/datasets/nvidia/Nemotron-Post-Training-Dataset-v2), which Puzzletron uses to score the quality of candidate block replacements during the NAS search.

In [ ]:
!cd /workspace/Model-Optimizer && \
python -m modelopt.torch.puzzletron.dataset.prepare_dataset \
    --dataset_name=nvidia/Nemotron-Post-Training-Dataset-v2 \
    --output_dir=/workspace/datasets/Nemotron-Post-Training-Dataset-v2

## 3. Configure the NAS search

In the YAML configuration files, we need to set:
- **`input_hf_model_path`**: path to the base Qwen3-8B model
- **`target_memory`**: set high (130,000 MiB) so it doesn't constrain — we're targeting by parameter count here
- **`num_params`**: 7B parameter target
- **`eval_samples`**: number of samples for scoring. A higher value can produce more reliable scores and potentially a better final architecture, but scoring time scales roughly linearly with this parameter. 32 is the value we use here as a reasonable accuracy/runtime trade-off for tutorial reproducibility.

In [ ]:
!sed -i 's|input_hf_model_path: .*|input_hf_model_path: /workspace/models/Qwen3-8B|' \
    /workspace/Model-Optimizer/examples/puzzletron/configs/qwen3-8b_pruneffn_memory/qwen3_8b_pruneffn_memory.yaml

!sed -i 's|target_memory: .*|target_memory: 130_000|' \
    /workspace/Model-Optimizer/examples/puzzletron/configs/qwen3-8b_pruneffn_memory/qwen3_8b_pruneffn_memory.yaml

!sed -i 's|target_memory: .*|target_memory: 130_000|' \
    /workspace/Model-Optimizer/examples/puzzletron/configs/qwen3-8b_pruneffn_memory/qwen3_8b.yaml

!sed -i 's|num_params: .*|num_params: 7_000_000_000|' \
    /workspace/Model-Optimizer/examples/puzzletron/configs/qwen3-8b_pruneffn_memory/qwen3_8b.yaml

!sed -i '/^scoring:/,/^[a-z]/{s|eval_samples: .*|eval_samples: 32|}' \
    /workspace/Model-Optimizer/examples/puzzletron/configs/qwen3-8b_pruneffn_memory/qwen3_8b.yaml

## 4. Run Puzzletron NAS search (Longest step: 5 hours at first run)

This step is significantly longer than Minitron's single-command pruning.

The MIP (Mixed-Integer Programming) solver will find the optimal heterogeneous architecture that has at most 7B parameters.

In [ ]:
# Remove if already exists from a previous run
!rm -f /workspace/puzzle_dir/subblock_stats.json
!cd /workspace/Model-Optimizer && \
torchrun --nproc_per_node 1 \
    examples/puzzletron/main.py \
    --config examples/puzzletron/configs/qwen3-8b_pruneffn_memory/qwen3_8b_pruneffn_memory.yaml \
    2>&1 | tee /workspace/puzzletron_qwen3_7B.log

## 5. Evaluate pruned model (before distillation)

Evaluate the Puzzletron-compressed model on MMLU. The model is heterogeneous (variable FFN widths per layer), so we use the `lm_eval_hf.py` script which supports this architecture.

In [ ]:
!sed -i 's/"torch\.bfloat16"/"bfloat16"/g' \
    /workspace/puzzle_dir/mip/puzzle_solutions/target_memory_130000MiB-num_params_7G/solutions--checkpoints/solution_0/config.json

!cd /workspace/Model-Optimizer && \
python examples/llm_eval/lm_eval_hf.py \
    --model hf \
    --model_args pretrained=/workspace/puzzle_dir/mip/puzzle_solutions/target_memory_130000MiB-num_params_7G/solutions--checkpoints/solution_0/,dtype=bfloat16,parallelize=True \
    --tasks mmlu \
    --num_fewshot 5 \
    --batch_size 4

## 6. Distill

Distill the heterogeneous Puzzletron model against the original Qwen3-8B teacher. Same distillation recipe as the Minitron notebooks: 100 iterations on [WikiText-103](https://huggingface.co/datasets/Salesforce/wikitext/tree/main/wikitext-103-v1).

The `distill.py` script handles both distillation and automatic export to HuggingFace format in one step.

Launch TensorBoard to monitor the distillation loss in real time. Open http://localhost:6006 in your browser once the distillation cell is running.

> **Tip:** In the TensorBoard settings (top-right gear icon), check **"Reload data"** so the charts update automatically as training progresses.

In [ ]:
import subprocess

subprocess.Popen(
    [
        "tensorboard",
        "--logdir",
        "/workspace/output/distill_output_puzzle_7B/tb_logs",
        "--port",
        "6006",
    ]
)
print("TensorBoard started at http://localhost:6006")

Now, let's run the distillation.
> **Expected runtime: ~20-30 minutes on 2x H200.**

In [ ]:
!torchrun --nproc_per_node=2 \
    /workspace/Model-Optimizer/examples/megatron_bridge/distill.py \
    --student_hf_path /workspace/puzzle_dir/mip/puzzle_solutions/target_memory_130000MiB-num_params_7G/solutions--checkpoints/solution_0 \
    --teacher_hf_path /workspace/models/Qwen3-8B \
    --data_paths 1.0 /workspace/datasets/wikitext-103-v1/wikitext-train_text \
    --output_dir /workspace/output/distill_output_puzzle_7B \
    --hf_export_path /workspace/output/distilled_Qwen3-8B-Puzzle-7B \
    --student_hf_model Qwen/Qwen3-8B \
    --seq_length 4096 \
    --tp_size 2 \
    --pp_size 1 \
    --mbs 1 \
    --gbs 4 \
    --train_iters 100 \
    --lr 0.0001 \
    --min_lr 1e-05 \
    --lr_warmup_iters 10 \
    --eval_interval 10 \
    --eval_iters 10 \
    --log_interval 1

Finally, kill tensorboard:

In [ ]:
subprocess.run(["pkill", "-f", "tensorboard"])

## 7. Evaluate distilled model

Compare with the Minitron result at the same parameter target (see [`scenario1_minitron.ipynb`](scenario1_minitron.ipynb)).

**Expected results on Qwen3-8B:**

| Model | Parameters | MMLU (5-shot) | % of Teacher |
|---|---|---|---|
| Qwen3-8B (teacher) | 8B | 0.7493 | 100% |
| **Minitron 7B — distilled** | **6.96B** | **0.7166** | **95.6%** |
| Puzzletron 7B — pruned | 6.99B | 0.6621 | 88.4% |
| Puzzletron 7B — distilled | 6.99B | 0.6823 | 91.1% |

In [ ]:
!cd /workspace/Model-Optimizer && \
python examples/llm_eval/lm_eval_hf.py \
    --model hf \
    --model_args pretrained=/workspace/output/distilled_Qwen3-8B-Puzzle-7B,dtype=bfloat16,parallelize=True \
    --tasks mmlu \
    --num_fewshot 5 \
    --batch_size 4